# Build and run CCT-S with TensorRT

Use the ONNX file from [the export notebook](04_export_onnx.ipynb). The Python runtime cells can be copied to another project. They require TensorRT, PyCUDA, NumPy, OpenCV and Pillow.

In [ ]:
from pathlib import Path
import os
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
USE_ENERGY_METER = os.environ.get("CCT_ENERGY_METER", "0") == "1"
WORK = ROOT / os.environ.get("CCT_WORK", "artifacts/energy_meter" if USE_ENERGY_METER else "artifacts")
ONNX_PATH = WORK / "onnx/best.onnx"
ENGINE_PATH = WORK / "model.engine"
DATASET = ROOT / ("dataset/energy-meter" if USE_ENERGY_METER else "dummy_data")
ALPHABET_OVERRIDE = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ._" if USE_ENERGY_METER else None

## Copyable runtime

The builder and engine execution use the same FP32 path as the repository scripts.

In [ ]:
"""TensorRT 10 explicit-batch runtime. No CUDA initialization on import."""
from pathlib import Path
import numpy as np

def build(onnx, engine, max_batch=32, fp16=False):
    import tensorrt as trt
    logger=trt.Logger(trt.Logger.WARNING)
    builder=trt.Builder(logger);network=builder.create_network(0)
    parser=trt.OnnxParser(network,logger)
    if not parser.parse(Path(onnx).read_bytes()):
        raise RuntimeError('\n'.join(str(parser.get_error(i)) for i in range(parser.num_errors)))
    if network.num_inputs!=1:raise ValueError('Expected one image input')
    inp=network.get_input(0);shape=list(inp.shape)
    if shape[1:] not in ([3,64,128],[64,128,3]):raise ValueError(f'Unsupported shape {shape}')
    cfg=builder.create_builder_config();cfg.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE,1<<30)
    cfg.clear_flag(trt.BuilderFlag.TF32)
    if fp16:cfg.set_flag(trt.BuilderFlag.FP16)
    if shape[0]<0:
        profile=builder.create_optimization_profile()
        profile.set_shape(inp.name,[1]+shape[1:],[min(8,max_batch)]+shape[1:],[max_batch]+shape[1:]);cfg.add_optimization_profile(profile)
    elif shape[0]!=1:raise ValueError('Only static B1 or dynamic batch supported')
    serialized=builder.build_serialized_network(network,cfg)
    if serialized is None:raise RuntimeError('TensorRT build failed')
    Path(engine).parent.mkdir(parents=True,exist_ok=True);Path(engine).write_bytes(bytes(serialized))

class Runtime:
    def __init__(self,path):
        import tensorrt as trt
        import pycuda.driver as cuda
        self.cuda=cuda;self.trt=trt;cuda.init();self.device_context=cuda.Device(0).make_context()
        self.allocations={}
        try:
            self.runtime=trt.Runtime(trt.Logger(trt.Logger.WARNING));self.engine=self.runtime.deserialize_cuda_engine(Path(path).read_bytes())
            if self.engine is None:raise RuntimeError('Invalid engine')
            self.context=self.engine.create_execution_context();self.stream=cuda.Stream()
            self.names=[self.engine.get_tensor_name(i) for i in range(self.engine.num_io_tensors)]
            self.input_name=next(n for n in self.names if self.engine.get_tensor_mode(n)==trt.TensorIOMode.INPUT)
            self.shape=tuple(self.engine.get_tensor_shape(self.input_name));self.dtype=np.dtype(trt.nptype(self.engine.get_tensor_dtype(self.input_name)))
        except Exception:
            self.close();raise
    def run(self,x):
        x=np.ascontiguousarray(x,dtype=self.dtype)
        if not self.context.set_input_shape(self.input_name,x.shape):raise ValueError(f'Invalid shape {x.shape}')
        outputs={}
        for n in self.names:
            shape=tuple(self.context.get_tensor_shape(n));dtype=np.dtype(self.trt.nptype(self.engine.get_tensor_dtype(n)))
            if any(v<0 for v in shape):raise ValueError(f'Unresolved shape {n}: {shape}')
            size=int(np.prod(shape))*dtype.itemsize
            if n not in self.allocations or self.allocations[n][1]<size:
                if n in self.allocations:self.allocations[n][0].free()
                self.allocations[n]=(self.cuda.mem_alloc(size),size)
            self.context.set_tensor_address(n,int(self.allocations[n][0]))
            if n!=self.input_name:outputs[n]=np.empty(shape,dtype)
        self.cuda.memcpy_htod(self.allocations[self.input_name][0],x)
        if not self.context.execute_async_v3(self.stream.handle):raise RuntimeError('Execution failed')
        self.stream.synchronize()
        for n,out in outputs.items():self.cuda.memcpy_dtoh(out,self.allocations[n][0])
        return list(outputs.values())
    def close(self):
        device_context = getattr(self, "device_context", None)
        if device_context is None:
            return
        try:
            if getattr(self, "stream", None) is not None:
                self.stream.synchronize()
            self.context = None
            self.engine = None
            self.runtime = None
            for allocation, _ in self.allocations.values():
                allocation.free()
            self.allocations.clear()
            self.stream = None
        finally:
            device_context.pop()
            device_context.detach()
            self.device_context = None

In [ ]:
import cv2
from PIL import Image

ALPHABET='0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ_'
def decode(outputs, alphabet=ALPHABET, pad="_"):
    values=list(outputs.values()) if isinstance(outputs,dict) else outputs if isinstance(outputs,(list,tuple)) else [outputs]
    candidates=[np.asarray(v) for v in values if len(v.shape)==3 and v.shape[-1]==len(alphabet)]
    if len(candidates)!=1:raise ValueError(f'Expected exactly one [B,slots,{len(alphabet)}] plate output')
    logits=candidates[0]
    return [''.join(alphabet[i] for i in row).split(pad,1)[0] for row in logits.argmax(-1)],logits

class CCTTensorRT(Runtime):
    def __init__(self, onnx_path, engine_path, *, max_batch=32, force_rebuild=False,
                 alphabet=ALPHABET, pad="_"):
        self.alphabet, self.pad = alphabet, pad
        if force_rebuild or not Path(engine_path).is_file():
            build(onnx_path, engine_path, max_batch=max_batch)
        super().__init__(engine_path)

    def predict_with_logits(self, images):
        pixels = np.stack([cv2.resize(im, (128, 64), interpolation=cv2.INTER_LINEAR)
                           for im in images]).astype(self.dtype)
        if self.shape[1] == 3:
            pixels = pixels.transpose(0, 3, 1, 2).copy()
        return decode(self.run(pixels), self.alphabet, self.pad)

    def predict(self, images):
        return self.predict_with_logits(images)[0]

## Build, read a crop, and compare against ONNX Runtime

In [ ]:
import csv
import onnxruntime as ort

with (DATASET / "val_annotations.csv").open(newline="") as stream:
    rows = list(csv.DictReader(stream))
images = []
for row in rows:
    with Image.open(DATASET / row["image_path"]) as image:
        images.append(np.asarray(image.convert("RGB")).copy())

if USE_ENERGY_METER:
    import yaml
    plate_spec = yaml.safe_load((ROOT / "config/cct_s_energy_meter_config.yaml").read_text())
    ALPHABET_OVERRIDE = plate_spec["alphabet"]

reader = CCTTensorRT(ONNX_PATH, ENGINE_PATH, alphabet=ALPHABET_OVERRIDE or ALPHABET)
reference = ort.InferenceSession(str(ONNX_PATH), providers=["CPUExecutionProvider"])
try:
    display(Image.fromarray(images[0]))
    for batch_size in (1, 2):
        predictions = []
        for start in range(0, len(images), batch_size):
            batch = images[start:start + batch_size]
            texts, logits = reader.predict_with_logits(batch)
            inp = reference.get_inputs()[0]
            dtype = np.uint8 if inp.type == "tensor(uint8)" else np.float32
            pixels = np.stack([cv2.resize(im, (128, 64), interpolation=cv2.INTER_LINEAR)
                               for im in batch]).astype(dtype)
            if inp.shape[1] == 3:
                pixels = pixels.transpose(0, 3, 1, 2).copy()
            expected, ort_logits = decode(reference.run(None, {inp.name: pixels}), reader.alphabet, reader.pad)
            assert texts == expected, (texts, expected)
            np.testing.assert_array_equal(logits.argmax(-1), ort_logits.argmax(-1))
            np.testing.assert_allclose(logits, ort_logits, rtol=0, atol=0.05)
            predictions.extend(texts)
        exact = sum(text == row["plate_text"] for text, row in zip(predictions, rows, strict=True))
        print(f"Batch {batch_size}: exact {exact}/{len(rows)}; ONNX strings match")
    print("First prediction:", repr(predictions[0]), "Target:", repr(rows[0]["plate_text"]))
finally:
    reader.close()